In [74]:
import numpy as np 
import pandas as pd

max_columns = "display.max_columns"
pd.set_option(max_columns, None)


##### 2. Load Split Data

In [75]:
train_df = pd.read_csv('titanic_train_raw.csv')
test_df = pd.read_csv('titanic_test_raw.csv')

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Train data shape: (712, 14)
Test data shape: (179, 14)


In [76]:
test_df.head(5)

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone,survived
0,3,male,24.0,2,0,24.1500,S,Third,man,True,NaN,Southampton,False,0
1,3,male,44.0,0,1,16.1000,S,Third,man,True,NaN,Southampton,False,0
2,3,male,22.0,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,True,1
3,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,False,0
4,3,female,NaN,1,0,15.5000,Q,Third,woman,False,NaN,Queenstown,False,1


In [77]:
##### 3. Separate X and y
target = "survived"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

# test set
X_test = test_df.drop(columns=[target])
y_test = test_df[target]

##### 4. Checking missing values in the dataset

In [78]:
print("missing values in train set:")
print(X_train.isnull().sum())
print("-"*50)
print("\nmissing values in test set:")
print(X_test.isnull().sum())

missing values in train set:
pclass           0
sex              0
age            137
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           553
embark_town      2
alone            0
dtype: int64
--------------------------------------------------

missing values in test set:
pclass           0
sex              0
age             40
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
deck           135
embark_town      0
alone            0
dtype: int64


In [79]:
print("missing values percentage in train set:")
print(X_train.isnull().mean() * 100)
print("-"*50)
print("\nmissing values percentage in test set:")
print(X_test.isnull().mean() * 100)

missing values percentage in train set:
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
--------------------------------------------------

missing values percentage in test set:
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.000000
class           0.000000
who             0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alone           0.000000
dtype: float64


##### 5. Demonstation: Why dropna (drop the rows) is not usually done in ML pipelines

In [80]:
print("Demonstration of dropna:")
print("original train shape:", X_train.shape)

# make sure to drop the corresponding rows in y_train, \ similarly for test data
dropped_demo = X_train.dropna()
print("Shape after dropna:", dropped_demo.shape)

Demonstration of dropna:
original train shape: (712, 13)
Shape after dropna: (141, 13)


##### observaton
####### dropna removed all rows that had even on missing value. This can throw away a lot of training examples. it also introdcues bias because the missingness pattern is not random. tHEREFOR, DROPNA IS SHOWN ONLY FOR DEMONSTRATION AND NOT USED IN REAL PIPELINWS

#### 6. dEMONSTRATION: Columns with high missing percentage

In [81]:
missing_threshold = 40

missing_percent_train = X_train.isna().mean() * 100
print("Percentage missing in each column:")
print(missing_percent_train)

print("columns with more that 40% threshold")
high_missing_cols = missing_percent_train[missing_percent_train > missing_threshold].index.to_list()
print(high_missing_cols)

Percentage missing in each column:
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
columns with more that 40% threshold
['deck']


In [82]:


# to drop entire column
# X_train = X_train.drop(columns = high_missing_cols)
# X_test = X_test.drop(columns = high_missing_cols)

#### 7. identify numerical and categorical columns

In [83]:
# cols_list = X_train.columns.to_list()
# print(cols_list)

num_cols = X_train.select_dtypes(np.number).columns.to_list()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(num_cols)
print(cat_cols)



['pclass', 'age', 'sibsp', 'parch', 'fare']
['sex', 'embarked', 'class', 'who', 'deck', 'embark_town']


/var/folders/85/_7ym_kd52wl9lwnxjnhk2mxc0000gn/T/ipykernel_72430/4007890871.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


##### 8. impute missing values the correct way (Train only)

In [84]:
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

##### 8. Numerical imputation

In [85]:
# Numeric_mean = {}
numeric_medians = {}

for col in num_cols:
    #mean_val = X_train_imputed[col].mean()
    median_val = X_train_imputed[col].median()
    numeric_medians[col] = median_val
    print(f"filling numreic column {col} with TRAIN median:", median_val)
    X_train_imputed[col] = X_train_imputed[col].fillna(median_val)
    X_test_imputed[col] = X_test_imputed[col].fillna(median_val)
 


filling numreic column pclass with TRAIN median: 3.0
filling numreic column age with TRAIN median: 28.5
filling numreic column sibsp with TRAIN median: 0.0
filling numreic column parch with TRAIN median: 0.0
filling numreic column fare with TRAIN median: 14.4542


##### 8. Categorical imputation (mode)

In [86]:
categoral_cols = {}

for col in cat_cols:
    mode_val = X_train_imputed[col].mode().iloc[0]
    categoral_cols[col] = mode_val
    print(f"filling categorical column {col} with TRAIN median:", mode_val)
    X_train_imputed[col] = X_train_imputed[col].fillna(mode_val)
    X_test_imputed[col] = X_test_imputed[col].fillna(mode_val)

filling categorical column sex with TRAIN median: male
filling categorical column embarked with TRAIN median: S
filling categorical column class with TRAIN median: Third
filling categorical column who with TRAIN median: man
filling categorical column deck with TRAIN median: C
filling categorical column embark_town with TRAIN median: Southampton


In [87]:
print(X_train_imputed.isna().sum())
print(X_test_imputed.isna().sum())

pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alone          0
dtype: int64
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alone          0
dtype: int64


In [89]:
train_imputed_df = X_train_imputed.copy()
test_imputed_df = X_test_imputed.copy()

train_imputed_df[target] = y_train.values
test_imputed_df[target] = y_test.values

train_imputed_df.to_csv("titanic_train_imputed.csv", index = False)
test_imputed_df.to_csv("titanic_test_imputed.csv", index=False)